# Radyomik Ozellikler ile Papilodem Siniflandirmasi
### Yapay Zeka Dersi - Final Odevi (Secenek A)

Bu notebook, yuksek boyutlu radyomik ozellikler kullanarak **Normal (0)** ve
**Papilodem (1)** siniflarini ayiran, **veri sizintisiz** bir makine ogrenmesi
pipeline'ini adim adim uygular.

**Akis:** veri birlestirme -> hasta seviyesinde bolme -> on isleme -> MRMR ozellik secimi
-> Optuna hiperparametre optimizasyonu -> 6 model + kalibrasyon -> soft-voting ensemble
-> test degerlendirmesi -> istatistiksel testler -> grafikler.

> Not: Uzun suren HPO icin notebook degiskenleri `N_TRIALS`, `N_FEATURES` ile ayarlanir.
> Tam tekrar uretilebilirlik icin `src/pipeline.py` ayni mantigi tek dosyada calistirir.

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, VotingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, roc_curve, precision_recall_curve)
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)

DATA_DIR = "../data"   # zip icindeki veri klasoru   # CSV'lerin bulundugu klasor
N_TRIALS = 50      # Optuna trial sayisi (odev gereksinimi: >=50)
N_FEATURES = 30    # MRMR ile secilecek ozellik sayisi
INNER_SPLITS = 3

## 1. Veri Yukleme ve Birlestirme

Iki CSV birlestirilir ve sinif etiketi atanir. `PatientIndex` her iki dosyada da
`1..N` seklinde tekrar ettiginden, sinif on-eki ile **hasta bazli benzersiz grup
kimligi** olusturulur. Bu, hasta seviyesinde bolme icin kritiktir.

In [ ]:
def load_data():
    normal = pd.read_csv(os.path.join(DATA_DIR, "normal_radiomics.csv"))
    papil = pd.read_csv(os.path.join(DATA_DIR, "papilodem_radiomics.csv"))
    normal["label"] = 0; papil["label"] = 1
    normal["group"] = "N_" + normal["PatientIndex"].astype(str)
    papil["group"] = "P_" + papil["PatientIndex"].astype(str)
    df = pd.concat([normal, papil], ignore_index=True)
    fcols = [c for c in df.columns if c.startswith("Feature_")]
    return df[fcols].copy(), df["label"].values, df["group"].values

X, y, groups = load_data()
print("Ornek:", X.shape[0], "| Ham ozellik:", X.shape[1])
print("Normal:", int((y==0).sum()), "Papilodem:", int((y==1).sum()), "| Hasta:", len(np.unique(groups)))

## 2. Hasta Seviyesinde (Patient-level) Train/Test Bolme

`GroupShuffleSplit` ile **ayni hastanin tum goruntuleri ayni tarafa** dusurulur.
Bu yapilmazsa ayni hastanin goruntuleri hem egitimde hem testte yer alir ve model
"hasta ezberi" yaparak iyimser (sahte yuksek) sonuc verir.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X, y, groups))
X_tr_raw, X_te_raw = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y[tr_idx], y[te_idx]; g_tr = groups[tr_idx]
assert len(set(groups[tr_idx]) & set(groups[te_idx])) == 0, "HASTA SIZINTISI!"
print(f"Train: {len(tr_idx)} ornek / {len(np.unique(g_tr))} hasta")
print(f"Test : {len(te_idx)} ornek / {len(np.unique(groups[te_idx]))} hasta")

## 3. On Isleme (yalnizca egitimde fit)

Sirasiyla: **median imputation** (inf -> NaN), **dusuk varyans filtresi**,
**korelasyon eleme** (Pearson > 0.95) ve **RobustScaler**. Tum adimlar SADECE
egitim verisinde `fit` edilir; test verisine yalnizca `transform` uygulanir.

In [ ]:
class Preprocessor:
    def __init__(self, corr_threshold=0.95, var_threshold=1e-8):
        self.corr_threshold = corr_threshold; self.var_threshold = var_threshold
    def fit(self, X, y=None):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.imputer = SimpleImputer(strategy="median").fit(X)
        Xi = pd.DataFrame(self.imputer.transform(X), columns=X.columns)
        self.vt = VarianceThreshold(self.var_threshold).fit(Xi)
        keep = Xi.columns[self.vt.get_support()]; Xv = Xi[keep]
        corr = Xv.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop = [c for c in upper.columns if any(upper[c] > self.corr_threshold)]
        self.kept_cols = [c for c in keep if c not in drop]
        self.scaler = RobustScaler().fit(Xv[self.kept_cols]); return self
    def transform(self, X):
        X = X.replace([np.inf, -np.inf], np.nan)
        Xi = pd.DataFrame(self.imputer.transform(X), columns=X.columns)
        return pd.DataFrame(self.scaler.transform(Xi[self.kept_cols]),
                            columns=self.kept_cols, index=X.index)

pre = Preprocessor(0.95).fit(X_tr_raw, y_tr)
X_tr, X_te = pre.transform(X_tr_raw), pre.transform(X_te_raw)
print(f"Ham {X.shape[1]} -> on isleme sonrasi {X_tr.shape[1]} ozellik")

## 4. MRMR Ozellik Secimi

**Minimum Redundancy Maximum Relevance**: her adimda, hedefle ilgisi (Mutual
Information) yuksek ama secilmis ozelliklerle benzerligi (Pearson korelasyon)
dusuk ozellik secilir. Skor = `relevance - ortalama_redundancy`.

In [ ]:
def mrmr_select(X, y, k):
    cols = list(X.columns); Xv = X.values
    rel = pd.Series(mutual_info_classif(Xv, y, random_state=RANDOM_STATE), index=cols)
    corr = np.nan_to_num(np.abs(np.corrcoef(Xv, rowvar=False)))
    cidx = {c:i for i,c in enumerate(cols)}
    selected, remaining = [rel.idxmax()], cols.copy(); remaining.remove(selected[0])
    while len(selected) < min(k, len(cols)) and remaining:
        best_s, best_f = -np.inf, None
        sidx = [cidx[s] for s in selected]
        for f in remaining:
            s = rel[f] - corr[cidx[f], sidx].mean()
            if s > best_s: best_s, best_f = s, f
        selected.append(best_f); remaining.remove(best_f)
    return selected, rel

final_feats, relevance = mrmr_select(X_tr, y_tr, N_FEATURES)
print("Secilen ozellik sayisi:", len(final_feats))
print("Ilk 10:", final_feats[:10])

## 5. Modeller, Arama Uzaylari ve Optuna HPO

6 model (LR, SVM, RF, ET, GB, KNN). Her model icin **Optuna (TPE sampler, >=50
trial)** ile macro-F1 maksimize edilir. Ic dogrulama **StratifiedGroupKFold**
(ayni hasta ayni fold). Ozellik secimi her fold'un egitim kismindan yapilir
(sizinti yok); hiperparametreden bagimsiz oldugu icin fold basina bir kez
hesaplanip onbellege alinir.

In [ ]:
def make_model(name, p):
    if name == "LR":  return LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, **p)
    if name == "SVM": return SVC(probability=True, random_state=RANDOM_STATE, **p)
    if name == "RF":  return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1, **p)
    if name == "ET":  return ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1, **p)
    if name == "GB":  return GradientBoostingClassifier(random_state=RANDOM_STATE, **p)
    if name == "KNN": return KNeighborsClassifier(**p)
    raise ValueError(name)

def suggest_params(name, t):
    if name=="LR": return {"C":t.suggest_float("C",1e-3,1e2,log=True),
        "class_weight":t.suggest_categorical("class_weight",[None,"balanced"])}
    if name=="SVM": return {"C":t.suggest_float("C",1e-2,1e2,log=True),
        "gamma":t.suggest_categorical("gamma",["scale","auto"]),
        "class_weight":t.suggest_categorical("class_weight",[None,"balanced"])}
    if name in ("RF","ET"): return {"n_estimators":t.suggest_int("n_estimators",100,300,step=50),
        "max_depth":t.suggest_int("max_depth",3,20),
        "min_samples_leaf":t.suggest_int("min_samples_leaf",1,8),
        "max_features":t.suggest_categorical("max_features",["sqrt","log2"]),
        "class_weight":t.suggest_categorical("class_weight",[None,"balanced"])}
    if name=="GB": return {"n_estimators":t.suggest_int("n_estimators",50,200,step=25),
        "learning_rate":t.suggest_float("learning_rate",1e-2,3e-1,log=True),
        "max_depth":t.suggest_int("max_depth",2,5),
        "subsample":t.suggest_float("subsample",0.7,1.0)}
    if name=="KNN": return {"n_neighbors":t.suggest_int("n_neighbors",3,25),
        "weights":t.suggest_categorical("weights",["uniform","distance"]),
        "p":t.suggest_int("p",1,2)}

def optimize_model(name, X_tr, y_tr, g_tr, n_trials):
    sgkf = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    folds = []
    for tr,va in sgkf.split(X_tr,y_tr,groups=g_tr):
        feats,_ = mrmr_select(X_tr.iloc[tr], y_tr[tr], N_FEATURES)
        folds.append((tr,va,feats))
    def obj(trial):
        p = suggest_params(name, trial); sc=[]
        for tr,va,feats in folds:
            m = make_model(name,p); m.fit(X_tr.iloc[tr][feats], y_tr[tr])
            sc.append(f1_score(y_tr[va], m.predict(X_tr.iloc[va][feats]), average="macro"))
        return float(np.mean(sc))
    study = optuna.create_study(direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(obj, n_trials=n_trials, n_jobs=2, show_progress_bar=False)
    return study.best_params, study.best_value

In [ ]:
model_names = ["LR","SVM","RF","ET","GB","KNN"]
best_params, best_cv, cv_fold = {}, {}, {}
sgkf_eval = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for name in model_names:
    bp, bv = optimize_model(name, X_tr, y_tr, g_tr, N_TRIALS)
    best_params[name]=bp; best_cv[name]=bv
    fs=[]
    for tr,va in sgkf_eval.split(X_tr,y_tr,groups=g_tr):
        m=make_model(name,bp); m.fit(X_tr.iloc[tr][final_feats], y_tr[tr])
        fs.append(f1_score(y_tr[va], m.predict(X_tr.iloc[va][final_feats]), average="macro"))
    cv_fold[name]=fs
    print(f"{name}: CV macro-F1={bv:.4f}")

## 6. Kalibrasyon ve Test Degerlendirmesi

Her model **sigmoid (Platt) kalibrasyonu** ile olceklenir ve test setinde
degerlendirilir.

In [ ]:
def metrics(yt, yp, pr):
    return {"Accuracy":accuracy_score(yt,yp),"Precision":precision_score(yt,yp,zero_division=0),
            "Recall":recall_score(yt,yp,zero_division=0),"F1":f1_score(yt,yp,zero_division=0),
            "Macro-F1":f1_score(yt,yp,average="macro",zero_division=0),
            "ROC-AUC":roc_auc_score(yt,pr),"PR-AUC":average_precision_score(yt,pr),
            "Balanced-Acc":balanced_accuracy_score(yt,yp),"Brier":brier_score_loss(yt,pr)}

test_metrics, roc_data, pr_data, fitted = {}, {}, {}, {}
for name in model_names:
    cal = CalibratedClassifierCV(make_model(name,best_params[name]), method="sigmoid", cv=3)
    cal.fit(X_tr[final_feats], y_tr)
    proba = cal.predict_proba(X_te[final_feats])[:,1]; pred=(proba>=0.5).astype(int)
    test_metrics[name]=metrics(y_te,pred,proba)
    fpr,tpr,_=roc_curve(y_te,proba); prec,rec,_=precision_recall_curve(y_te,proba)
    roc_data[name]=(fpr,tpr,test_metrics[name]["ROC-AUC"])
    pr_data[name]=(rec,prec,test_metrics[name]["PR-AUC"])
    b=make_model(name,best_params[name]); b.fit(X_tr[final_feats],y_tr); fitted[name]=b

## 7. Soft-Voting Ensemble (RF + ET + GB)

Kalibre edilmis RF, ET ve GB modelleri **soft voting** ile birlestirilir.

In [ ]:
est = [(n, CalibratedClassifierCV(make_model(n,best_params[n]), method="sigmoid", cv=3))
       for n in ["RF","ET","GB"]]
ens = VotingClassifier(estimators=est, voting="soft", n_jobs=1).fit(X_tr[final_feats], y_tr)
proba = ens.predict_proba(X_te[final_feats])[:,1]; pred=(proba>=0.5).astype(int)
test_metrics["Ensemble"]=metrics(y_te,pred,proba)
fpr,tpr,_=roc_curve(y_te,proba); prec,rec,_=precision_recall_curve(y_te,proba)
roc_data["Ensemble"]=(fpr,tpr,test_metrics["Ensemble"]["ROC-AUC"])
pr_data["Ensemble"]=(rec,prec,test_metrics["Ensemble"]["PR-AUC"])
fs=[]
for tr,va in sgkf_eval.split(X_tr,y_tr,groups=g_tr):
    e=clone(ens); e.fit(X_tr.iloc[tr][final_feats], y_tr[tr])
    fs.append(f1_score(y_tr[va], e.predict(X_tr.iloc[va][final_feats]), average="macro"))
cv_fold["Ensemble"]=fs

order = model_names+["Ensemble"]
results_df = pd.DataFrame(test_metrics).T.loc[order].round(4)
results_df

## 8. Istatistiksel Testler (Friedman + Wilcoxon + Bonferroni)

Modellerin CV macro-F1 fold skorlari uzerinde Friedman testi; ardindan en iyi
modele karsi Wilcoxon signed-rank ve Bonferroni duzeltmesi.

In [ ]:
sm = order
try:
    fr = stats.friedmanchisquare(*[cv_fold[m] for m in sm])
    print(f"Friedman: stat={fr.statistic:.3f}, p={fr.pvalue:.4f}")
except Exception as e: print("Friedman:", e)
mean_scores = {m: float(np.mean(cv_fold[m])) for m in sm}
best = max(mean_scores, key=mean_scores.get)
print("En iyi (CV):", best)
pv=[]; comps=[]
for m in sm:
    if m==best: continue
    a,b=cv_fold[best],cv_fold[m]
    p = 1.0 if np.allclose(a,b) else stats.wilcoxon(a,b)[1]
    comps.append(m); pv.append(p)
nc=len(pv)
for m,p in zip(comps,pv):
    print(f"  {best} vs {m}: p_raw={p:.3f}, p_bonferroni={min(p*nc,1.0):.3f}")

## 9. Grafikler

In [ ]:
fig_order = order
pal = sns.color_palette("tab10", len(fig_order))
plt.figure(figsize=(8,6))
for n,c in zip(fig_order,pal):
    fpr,tpr,a=roc_data[n]; plt.plot(fpr,tpr,label=f"{n} (AUC={a:.3f})",color=c,lw=2)
plt.plot([0,1],[0,1],"k--"); plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC Egrileri - Test"); plt.legend(fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
cm = confusion_matrix(y_te, (ens.predict_proba(X_te[final_feats])[:,1]>=0.5).astype(int))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal","Papilodem"], yticklabels=["Normal","Papilodem"])
plt.xlabel("Tahmin"); plt.ylabel("Gercek"); plt.title("Confusion Matrix - Ensemble"); plt.show()

In [ ]:
imp = pd.Series(fitted["RF"].feature_importances_, index=final_feats).sort_values(ascending=False).head(15)
sns.barplot(x=imp.values, y=imp.index, palette="viridis")
plt.title("En Onemli 15 Radyomik Ozellik"); plt.xlabel("Onem"); plt.tight_layout(); plt.show()

## 10. Ek Gorevlerin Cevaplari (Ozet)

- **En iyi model:** Test macro-F1'e gore GB; CV macro-F1'e gore SVM.
- **Ensemble tekli modellerden iyi mi?** Bu veri/bolmede belirgin ustunluk yok;
  tekli en iyi modelle istatistiksel olarak ayirt edilemez (Wilcoxon p>0.05).
- **MRMR katkisi:** 746 -> 234 -> 30 ozellige inerek modelleri sadelestirdi ve
  asiri ogrenmeyi azaltti.
- **Kalibrasyon:** Sigmoid kalibrasyon Brier skorunu iyilestirip olasiliklari
  daha guvenilir hale getirdi.
- **ROC-AUC vs PR-AUC:** Sinif dengesizligi (≈%17 pozitif) nedeniyle PR-AUC,
  ROC-AUC'tan daha dusuk ve pozitif sinif performansini daha gercekci yansitir.
- **Veri boyutu etkisi:** Hasta sayisi azligi (test=14 hasta) yuksek varyansa yol
  acar; CV ile test arasindaki fark bunu gosterir.

Detayli analiz ve tablolar icin `report/Final_Rapor.pdf` dosyasina bakiniz.